# 3D CT classification — interactive lab

Train a **3D convolutional neural network** to tell apart CT scans **with** and **without**
COVID-related findings (viral pneumonia), then explore the model interactively:

1. build and train the network with sliders for the main hyper-parameters,
2. see the training curves and the ROC beside the controls,
3. sweep the **classification threshold** and watch sensitivity, specificity and the
   misclassified cases change,
4. browse individual 3D volumes slice by slice with the prediction overlaid.

Dataset: a subset of **MosMedData** (Chest CT with COVID-19 findings).
Based on the Keras example by Hasib Zunair.

## 1 · Setup and data

In [ ]:
#@title Install dependencies (quiet)
!pip install -q nibabel ipywidgets
import os, zipfile
import numpy as np
import tensorflow as tf
import keras
from keras import layers
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception:
    pass
print("TensorFlow", tf.__version__)

In [ ]:
#@title Download the MosMedData subset
# normal scans
keras.utils.get_file("CT-0.zip",
    "https://github.com/hasibzunair/3D-image-classification-tutorial/releases/download/v0.2/CT-0.zip",
    cache_dir=os.getcwd())
# abnormal scans
keras.utils.get_file("CT-23.zip",
    "https://github.com/hasibzunair/3D-image-classification-tutorial/releases/download/v0.2/CT-23.zip",
    cache_dir=os.getcwd())

os.makedirs("MosMedData", exist_ok=True)
# get_file stores under <cache_dir>/datasets/
for z in ("CT-0.zip", "CT-23.zip"):
    path = os.path.join(os.getcwd(), "datasets", z)
    with zipfile.ZipFile(path, "r") as zf:
        zf.extractall("./MosMedData/")
print("Extracted to ./MosMedData/")

## 2 · Read and preprocess the volumes
CT voxel intensities are in Hounsfield units; we clip to [-1000, 400] HU, normalise to [0,1], and resample every scan to 128×128×64.

In [ ]:

import nibabel as nib

from scipy import ndimage


def read_nifti_file(filepath):
    """Read and load volume"""
    # Read file
    scan = nib.load(filepath)
    # Get raw data
    scan = scan.get_fdata()
    return scan


def normalize(volume):
    """Normalize the volume"""
    min = -1000
    max = 400
    volume[volume < min] = min
    volume[volume > max] = max
    volume = (volume - min) / (max - min)
    volume = volume.astype("float32")
    return volume


def resize_volume(img):
    """Resize across z-axis"""
    # Set the desired depth
    desired_depth = 64
    desired_width = 128
    desired_height = 128
    # Get current depth
    current_depth = img.shape[-1]
    current_width = img.shape[0]
    current_height = img.shape[1]
    # Compute depth factor
    depth = current_depth / desired_depth
    width = current_width / desired_width
    height = current_height / desired_height
    depth_factor = 1 / depth
    width_factor = 1 / width
    height_factor = 1 / height
    # Rotate
    img = ndimage.rotate(img, 90, reshape=False)
    # Resize across z-axis
    img = ndimage.zoom(img, (width_factor, height_factor, depth_factor), order=1)
    return img


def process_scan(path):
    """Read and resize volume"""
    # Read scan
    volume = read_nifti_file(path)
    # Normalize
    volume = normalize(volume)
    # Resize width, height and depth
    volume = resize_volume(volume)
    return volume


In [ ]:
# Folder "CT-0" consist of CT scans having normal lung tissue,
# no CT-signs of viral pneumonia.
normal_scan_paths = [
    os.path.join(os.getcwd(), "MosMedData/CT-0", x)
    for x in os.listdir("MosMedData/CT-0")
]
# Folder "CT-23" consist of CT scans having several ground-glass opacifications,
# involvement of lung parenchyma.
abnormal_scan_paths = [
    os.path.join(os.getcwd(), "MosMedData/CT-23", x)
    for x in os.listdir("MosMedData/CT-23")
]

print("CT scans with normal lung tissue: " + str(len(normal_scan_paths)))
print("CT scans with abnormal lung tissue: " + str(len(abnormal_scan_paths)))


In [ ]:
# Read and process the scans.
# Each scan is resized across height, width, and depth and rescaled.
abnormal_scans = np.array([process_scan(path) for path in abnormal_scan_paths])
normal_scans = np.array([process_scan(path) for path in normal_scan_paths])

# For the CT scans having presence of viral pneumonia
# assign 1, for the normal ones assign 0.
abnormal_labels = np.array([1 for _ in range(len(abnormal_scans))])
normal_labels = np.array([0 for _ in range(len(normal_scans))])

# Split data in the ratio 70-30 for training and validation.
x_train = np.concatenate((abnormal_scans[:70], normal_scans[:70]), axis=0)
y_train = np.concatenate((abnormal_labels[:70], normal_labels[:70]), axis=0)
x_val = np.concatenate((abnormal_scans[70:], normal_scans[70:]), axis=0)
y_val = np.concatenate((abnormal_labels[70:], normal_labels[70:]), axis=0)
print(
    "Number of samples in train and validation are %d and %d."
    % (x_train.shape[0], x_val.shape[0])
)

## 3 · Augmentation and data loaders
Training volumes are randomly rotated on the fly; a channel axis is added for 3D convolution.

In [ ]:
import random

from scipy import ndimage


def rotate(volume):
    """Rotate the volume by a few degrees"""

    def scipy_rotate(volume):
        # define some rotation angles
        angles = [-20, -10, -5, 5, 10, 20]
        # pick angles at random
        angle = random.choice(angles)
        # rotate volume
        volume = ndimage.rotate(volume, angle, reshape=False)
        volume[volume < 0] = 0
        volume[volume > 1] = 1
        return volume

    augmented_volume = tf.numpy_function(scipy_rotate, [volume], tf.float32)
    return augmented_volume


def train_preprocessing(volume, label):
    """Process training data by rotating and adding a channel."""
    # Rotate volume
    volume = rotate(volume)
    volume = tf.expand_dims(volume, axis=3)
    return volume, label


def validation_preprocessing(volume, label):
    """Process validation data by only adding a channel."""
    volume = tf.expand_dims(volume, axis=3)
    return volume, label


In [ ]:
# Define data loaders.
train_loader = tf.data.Dataset.from_tensor_slices((x_train, y_train))
validation_loader = tf.data.Dataset.from_tensor_slices((x_val, y_val))

batch_size = 2
# Augment the on the fly during training.
train_dataset = (
    train_loader.shuffle(len(x_train))
    .map(train_preprocessing)
    .batch(batch_size)
    .prefetch(2)
)
# Only rescale.
validation_dataset = (
    validation_loader.shuffle(len(x_val))
    .map(validation_preprocessing)
    .batch(batch_size)
    .prefetch(2)
)

## 4 · Model factory
The 3D CNN is built by a function so the interactive panel can rebuild it with different widths and dropout.

In [ ]:
def get_model(width=128, height=128, depth=64, base_filters=64, dropout=0.3):
    """Build a 3D CNN. base_filters and dropout are exposed to the sliders."""
    inputs = keras.Input((width, height, depth, 1))
    f = base_filters
    x = layers.Conv3D(filters=f,   kernel_size=3, activation="relu")(inputs)
    x = layers.MaxPool3D(pool_size=2)(x); x = layers.BatchNormalization()(x)
    x = layers.Conv3D(filters=f,   kernel_size=3, activation="relu")(x)
    x = layers.MaxPool3D(pool_size=2)(x); x = layers.BatchNormalization()(x)
    x = layers.Conv3D(filters=f*2, kernel_size=3, activation="relu")(x)
    x = layers.MaxPool3D(pool_size=2)(x); x = layers.BatchNormalization()(x)
    x = layers.Conv3D(filters=f*4, kernel_size=3, activation="relu")(x)
    x = layers.MaxPool3D(pool_size=2)(x); x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(units=512, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(units=1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="3dcnn")

## 5 · Interactive training

Set the hyper-parameters with the sliders, then press **Train**. The training curves and
the ROC on the validation set appear beside the controls. Everything runs on the small
subset, so a few epochs take a minute or two on a GPU runtime.

*Tip: a 3D CNN is heavy — start with few epochs and small base-filters to see it work,
then scale up.*

In [ ]:
#@title 🎛 Interactive training — build, train, ROC beside the sliders
import time, datetime
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

# store the trained model + validation scores for the later cells
STATE = {"model": None, "y_true": None, "y_score": None, "history": None}

name_w    = W.Text(value="", description="name")
epochs_w  = W.IntSlider(value=15, min=3, max=60, step=1, description="epochs", continuous_update=False)
filters_w = W.Dropdown(options=[16, 32, 64], value=32, description="base filters")
drop_w    = W.FloatSlider(value=0.3, min=0.0, max=0.6, step=0.1, description="dropout", continuous_update=False)
lr_w      = W.Dropdown(options=[1e-4, 3e-4, 1e-3], value=1e-4, description="lr")
arch_l    = W.HTML()
train_btn = W.Button(description="Train", button_style="success", icon="play")
status    = W.HTML()
out       = W.Output()

def update_arch(*_):
    f = filters_w.value
    chain = " → ".join(["Conv%d"%f, "Conv%d"%f, "Conv%d"%(f*2), "Conv%d"%(f*4),
                        "GAP", "Dense512", "Drop%.1f"%drop_w.value, "Dense1 (sigmoid)"])
    arch_l.value = "<b>architecture:</b> " + chain
for wdg in (filters_w, drop_w): wdg.observe(update_arch, "value")
update_arch()

def on_train(_):
    train_btn.disabled = True
    with out:
        clear_output(wait=True)
        try:
            keras.backend.clear_session()
            model = get_model(base_filters=filters_w.value, dropout=drop_w.value)
            lr_sched = keras.optimizers.schedules.ExponentialDecay(
                lr_w.value, decay_steps=100000, decay_rate=0.96, staircase=True)
            model.compile(loss="binary_crossentropy",
                          optimizer=keras.optimizers.Adam(learning_rate=lr_sched),
                          metrics=["acc"])
            print("Training {} epochs | base filters {} | dropout {} | lr {}".format(epochs_w.value, filters_w.value, drop_w.value, lr_w.value))
            t1 = time.perf_counter()
            history = model.fit(train_dataset, validation_data=validation_dataset,
                                epochs=epochs_w.value, verbose=2)
            secs = time.perf_counter() - t1
        except Exception as e:
            print("Training failed:", e); train_btn.disabled = False; return

        # ROC on the validation set
        y_score = model.predict(validation_dataset, verbose=0).ravel()
        y_true  = np.concatenate([y.numpy() for _, y in validation_dataset], axis=0)
        fpr, tpr, thr = roc_curve(y_true, y_score); roc_auc = auc(fpr, tpr)
        STATE.update(model=model, y_true=y_true, y_score=y_score, history=history)

        fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
        ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="valid")
        ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
        ax[1].plot(history.history["acc"], label="train"); ax[1].plot(history.history["val_acc"], label="valid")
        ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
        plt.tight_layout(); plt.show()

        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, lw=2.5, color="crimson", label="AUC = %.3f" % roc_auc)
        plt.plot([0, 1], [0, 1], "--", color="gray", lw=1)
        plt.xlabel("False Positive Rate (1 − specificity)")
        plt.ylabel("True Positive Rate (sensitivity)")
        plt.title("Validation ROC — pneumonia vs normal")
        plt.legend(loc="lower right"); plt.grid(alpha=.3); plt.show()
        print("AUC = {:.3f} | final val acc = {:.3f} | {:.0f}s".format(roc_auc, history.history["val_acc"][-1], secs))
        status.value = "<span style='color:green'>Done ✓ — now use the threshold cell below.</span>"
    train_btn.disabled = False

train_btn.on_click(on_train)
controls = W.VBox([name_w, epochs_w, filters_w, drop_w, lr_w, arch_l, train_btn, status])
display(W.HBox([controls, out]))

## 6 · Tune the classification threshold

The network outputs a probability of pneumonia. **You** decide the cut-off that turns that
probability into a yes/no. Move the slider and watch how sensitivity, specificity and the
**number of misclassified scans** change — a low threshold catches more disease but raises
false positives; a high threshold does the opposite.

In [ ]:
#@title 🎚 Threshold vs sensitivity / specificity / misclassified
import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def show_threshold(thr):
    if STATE["model"] is None:
        print("Train the model in the cell above first."); return
    y_true, y_score = STATE["y_true"], STATE["y_score"]
    y_pred = (y_score >= thr).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0
    mis  = fp + fn

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    # confusion matrix
    cm = np.array([[tn, fp], [fn, tp]])
    ax[0].imshow(cm, cmap="Oranges")
    for (i, j), v in np.ndenumerate(cm):
        ax[0].text(j, i, str(v), ha="center", va="center", fontsize=16,
                   color="white" if v > cm.max()/2 else "black")
    ax[0].set_xticks([0,1]); ax[0].set_xticklabels(["pred normal","pred pneumonia"])
    ax[0].set_yticks([0,1]); ax[0].set_yticklabels(["true normal","true pneumonia"])
    ax[0].set_title("Confusion matrix @ threshold %.2f" % thr)
    # probability histogram with the threshold line
    ax[1].hist(y_score[y_true==0], bins=15, alpha=.6, label="normal", color="steelblue")
    ax[1].hist(y_score[y_true==1], bins=15, alpha=.6, label="pneumonia", color="crimson")
    ax[1].axvline(thr, color="black", ls="--", lw=2, label="threshold")
    ax[1].set_xlabel("predicted probability of pneumonia"); ax[1].set_ylabel("scans")
    ax[1].set_title("Score distribution"); ax[1].legend()
    plt.tight_layout(); plt.show()
    print("Sensitivity = {:.2f}   Specificity = {:.2f}   Misclassified = {} of {}".format(sens, spec, mis, len(y_true)))
    print("TP={}  TN={}  FP={}  FN={}".format(tp, tn, fp, fn))

thr_w = W.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.02,
                      description="threshold", continuous_update=False,
                      layout=W.Layout(width="500px"))
W.interact(show_threshold, thr=thr_w)

## 7 · Inspect the misclassified scans

Where does the model actually fail? Pick a threshold, then step through the scans it got
wrong and scroll the slices. False negatives are often subtle or localised findings; false
positives are often motion or reconstruction artefacts.

In [ ]:
#@title 🔎 Browse the errors slice by slice
import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import display

def browse_errors(thr, which, error_idx, z):
    if STATE["model"] is None:
        print("Train the model first."); return
    y_true, y_score = STATE["y_true"], STATE["y_score"]
    y_pred = (y_score >= thr).astype(int)
    if which == "false negatives":
        idxs = np.where((y_pred == 0) & (y_true == 1))[0]
    elif which == "false positives":
        idxs = np.where((y_pred == 1) & (y_true == 0))[0]
    else:
        idxs = np.where(y_pred == y_true)[0]
    if len(idxs) == 0:
        print("None of these at threshold %.2f." % thr); return
    k = idxs[error_idx % len(idxs)]
    vol = x_val[k]                       # (128,128,64)
    z = min(z, vol.shape[2]-1)
    plt.figure(figsize=(5,5))
    plt.imshow(vol[:, :, z], cmap="gray")
    plt.title("scan #{} — true={}  pred prob={:.2f}\nslice {}/{}".format(k, "pneumonia" if y_true[k] else "normal", y_score[k], z, vol.shape[2]-1))
    plt.axis("off"); plt.show()

W.interact(browse_errors,
    thr=W.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="threshold", continuous_update=False),
    which=W.Dropdown(options=["false negatives","false positives","correct"], value="false negatives", description="show"),
    error_idx=W.IntSlider(value=0, min=0, max=20, step=1, description="case #", continuous_update=False),
    z=W.IntSlider(value=32, min=0, max=63, step=1, description="slice", continuous_update=False))

## 8 · Predict on a single scan

In [ ]:
if STATE["model"] is not None:
    prediction = STATE["model"].predict(np.expand_dims(x_val[0], axis=0), verbose=0)[0]
    scores = [1 - prediction[0], prediction[0]]
    for score, name in zip(scores, ["normal", "abnormal"]):
        print("This model is {:.1f}% confident that the CT scan is {}".format(100*score, name))
else:
    print("Train the model first (cell 5).")